In [ ]:
import pandas as pd

In [ ]:
OzOFF_database_path="lipid_database/OzOFF_database/Database_OzOFF.parquet"
OzON_database_path = "lipid_database/OzON_databases/OzON_Possible_Database_0.parquet"
df_OFF = pd.read_parquet(OzOFF_database_path)
df_ON = pd.read_parquet(OzON_database_path)
df_OFF

In [ ]:
df_ON

# OFF 

# 1 - mzml parser

In [ ]:
# Submit the SLURM job for mzml_parser_1.sh
!sbatch core/backend/mzml_parser_1_AMP_OFF.sh


In [ ]:
df = pd.read_parquet('Projects/AMP/mzml_parsed/OFF/df_mzml_parser_1_OFF.parquet')
df

In [ ]:
transitions = pd.read_parquet('Projects/AMP/mzml_parsed/OFF/df_transition_summed_1_OFF.parquet')
transitions

# 2 - sample name 

In [ ]:
# Submit the SLURM job for !sbatch core/backend/sample_2.sh
!sbatch core/backend/sample_2_AMP_OFF.sh


In [ ]:

df = pd.read_parquet('Projects/AMP/samples/OFF/df_sample_2_cereb_WT_m1_FAD245.parquet')
df

# 3 - Match lipids NOT POSSIBLE

count # of files and set this number in the .sh file

In [ ]:
input_dir = 'Projects/AMP/samples/OFF/'
import glob; num_files = len(glob.glob('Projects/AMP/samples/OFF/*.parquet'))
num_files   


In [ ]:
# Submit the SLURM job for !sbatch core/backend/match_3.sh
!sbatch core/backend/not_possible/match_3_AMP_OFF_notpossible.sh



In [ ]:
import pandas as pd
df = pd.read_parquet('Projects/AMP/match/OFF/notpossible/df_match_3_hippo_WT_m1_FAD245.parquet')
print(df['Lipid'].unique())
print(df['Lipid'].unique())

df

# 4 - group

In [ ]:
!sbatch core/backend/not_possible/group_4_AMP_OFF_notpossible.sh

In [ ]:
import pandas as pd

# Load the parquet file
df_before = pd.read_parquet('Projects/AMP/group/OFF/notpossible/df_group_4_hippo_WT_m1_FAD245_OFF.parquet')

# # Filter out the rows where Lipid is 'FA(24:1)' and Parent_Ion is 533.5
# filtered_df = df_before[~((df_before['Lipid'] == 'FA(24:1)') & (df_before['Parent_Ion'] == 533.5))]

# # # Save the filtered DataFrame
# # filtered_df.to_parquet('Projects/STD/group/OFF/df_group_4_FAME_OFF_filtered24.parquet')

# # Display the filtered DataFrame
# filtered_df

print(df_before['Lipid'].unique())
print(df_before['Species'].unique())
df_before


In [ ]:
notpossible_lipids = df_before[df_before['Species'].str.contains('16:1', case=False, na=False)]
sorted_notpossible_lipids = notpossible_lipids.sort_values(by='OzESI_Intensity', ascending=False)
#print unique Lipid values
# print(sorted_notpossible_lipids['Species'].unique())
sorted_notpossible_lipids


In [ ]:
import matplotlib.pyplot as plt

# Function to plot chromatograms, with options to either display or save the plots
def plot_chromatograms(df, save_path=None, show_only=False, specific_group=None):
    if specific_group:
        # If a specific group is provided, filter the DataFrame for that group only
        group_data = df[df['group_by_lipid'] == specific_group]
        
        # Ensure the Retention_Time is sorted
        group_data = group_data.sort_values(by='Retention_Time')
        
        # Extract Lipid and Parent Ion values from the first row of the group
        lipid = group_data['Lipid'].iloc[0] if not group_data.empty else f"Unknown_Lipid_{specific_group}"
        parent_ion = group_data['Parent_Ion'].iloc[0] if not group_data.empty else "Unknown Parent Ion"
        
        plt.figure(figsize=(10, 6))
        plt.plot(group_data['Retention_Time'], group_data['OzESI_Intensity'], label=f'group_by_lipid: {specific_group}')
        plt.xlabel('Retention Time')
        plt.ylabel('OzESI Intensity')
        
        # Add Lipid and Parent Ion to the title
        plt.title(f'{specific_group}  | Parent Ion: {parent_ion}')
        
        plt.legend()
        plt.grid(True)
        
        if show_only:
            plt.show()  # Display the plot
        else:
            # Save the plot as a PNG with the lipid name followed by "_OFF"
            save_file = f"{save_path}/{lipid.replace(':', '_').replace('/', '_')}_OFF.png"  # Clean filename
            plt.savefig(save_file)
            print(f"Plot saved as {save_file}")
        plt.close()  # Close the figure after displaying or saving

    else:
        # Iterate through each unique group_by_lipid value if no specific group is provided
        for group_value in df['group_by_lipid'].unique():
            group_data = df[df['group_by_lipid'] == group_value]
            
            # Ensure the Retention_Time is sorted
            group_data = group_data.sort_values(by='Retention_Time')
            
            # Extract Lipid and Parent Ion values from the first row of the group
            lipid = group_data['Lipid'].iloc[0] if not group_data.empty else f"Unknown_Lipid_{group_value}"
            parent_ion = group_data['Parent_Ion'].iloc[0] if not group_data.empty else "Unknown Parent Ion"
            
            plt.figure(figsize=(10, 6))
            plt.plot(group_data['Retention_Time'], group_data['OzESI_Intensity'], label=f'group_by_lipid: {group_value}')
            plt.xlabel('Retention Time')
            plt.ylabel('OzESI Intensity')
            
            # Add Lipid and Parent Ion to the title
            plt.title(f'{group_value} | | Parent Ion: {parent_ion}')
            
            plt.legend()
            plt.grid(True)
            
            if show_only:
                plt.show()  # Display the plot
                break  # Exit after showing one plot if in "show only" mode
            else:
                # Save the plot as a PNG with the lipid name followed by "_OFF"
                save_file = f"{save_path}/{lipid.replace(':', '_').replace('/', '_')}_OFF.png"  # Clean filename
                plt.savefig(save_file)
                print(f"Plot saved as {save_file}")
            plt.close()  # Close the figure to avoid overlap in the next iteration

# Example usage:
# User-defined variable for the save path
save_path = 'Projects/AMP/plots/OFF_plots/'  # Replace with your desired save path

# # Call the function to either display or save chromatograms
# # For saving and iterating through all groups:
# plot_chromatograms(df, save_path=save_path)

# For showing only a specific group without saving:
plot_chromatograms(df_before, show_only=True, specific_group=33)


# 5 - Analysis

In [ ]:
!sbatch core/backend/not_possible/analysis_5_AMP_OFF_notpossible.sh

In [ ]:
# import pandas as pd
# df_analysis = pd.read_parquet('Projects/STD/analysis/OFF/multiple_values/df_analysis_5_FAME_OFF.parquet')
#  #Projects/STD/analysis/OFF/df_analysis_5_FAME_OFF.parquet
# print(df_analysis['Lipid'].unique())
# df_analysis.head(10)
import pandas as pd
df_analysis = pd.read_parquet('Projects/AMP/analysis/OFF/notpossible/df_analysis_5_hippo_WT_m1_FAD245_OFF.parquet')
# pd.read_parquet('core/python/')
# df_analysis = df_analysis[(df_analysis['Retention_Time'] >= 6.4) & (df_analysis['Retention_Time'] <= 7.9)]
df_analysis

# plot analysis and group

In [ ]:
import matplotlib.pyplot as plt

def plot_chromatogram(df_grouped, df_analysis, group_value, retention_start=None, retention_stop=None):
    """
    Plot the chromatogram for a specific group_by_lipid value with an optional retention time range.

    :param df_grouped: DataFrame containing the original grouped data.
    :param df_analysis: DataFrame containing the analyzed data (peaks).
    :param group_value: The specific group_by_lipid value to plot.
    :param retention_start: The start of the retention time range (inclusive). Default is None.
    :param retention_stop: The end of the retention time range (inclusive). Default is None.
    """
    # Filter data for the specific group_by_lipid value
    group_data = df_grouped[df_grouped['group_by_lipid'] == group_value]
    analysis_data = df_analysis[df_analysis['group_by_lipid'] == group_value]
    
    # Apply the retention time range filter if specified
    if retention_start is not None and retention_stop is not None:
        group_data = group_data[(group_data['Retention_Time'] >= retention_start) & 
                                (group_data['Retention_Time'] <= retention_stop)]
        analysis_data = analysis_data[(analysis_data['Retention_Time'] >= retention_start) & 
                                      (analysis_data['Retention_Time'] <= retention_stop)]
    
    # Ensure the Retention_Time is sorted in both DataFrames
    group_data = group_data.sort_values(by='Retention_Time')
    analysis_data = analysis_data.sort_values(by='Retention_Time')

    # Extract Lipid, Parent Ion, and Isomer values from the first row of the group
    lipid = group_data['Lipid'].iloc[0] if not group_data.empty else "Unknown Lipid"
    parent_ion = group_data['Parent_Ion'].iloc[0] if not group_data.empty else "Unknown Parent Ion"
    
    # Plot the grouped data (original chromatogram)
    plt.figure(figsize=(10, 6))
    plt.plot(group_data['Retention_Time'], group_data['OzESI_Intensity'], 
             label=f'Chromatogram - group_by_lipid: {group_value}', color='blue')
    
    # Overlay the analysis data (peaks)
    if not analysis_data.empty:
        plt.scatter(analysis_data['Retention_Time'], analysis_data['OzESI_Intensity'], 
                    color='red', marker='x', label='Identified Peaks')

    # Add labels and title
    plt.xlabel('Retention Time')
    plt.ylabel('OzESI Intensity')
    
    # Add Lipid, Parent Ion, and Isomer to the title
    title = f'{group_value} | Parent Ion: {parent_ion} OzOFF NOT POSSIBLE'
    plt.title(title)
    
    # Add a legend and grid
    plt.legend()
    plt.grid(True)
    
    # Show the plot
    plt.show()

# Example: Define the specific group_by_lipid value and retention time range
specific_group_by_lipid_value = 1  # Replace with the actual value you are interested in
retention_time_start = 6  # Replace with your desired start time
retention_time_stop = 8.5  # Replace with your desired stop time

# Call the function to plot the chromatogram and analysis data
plot_chromatogram(df_before, df_analysis, specific_group_by_lipid_value, 
                  retention_start=retention_time_start, retention_stop=retention_time_stop)


# STD RT Adjustment

In [ ]:
def STD_RT_Adjustment(ozon_df, fame_std):
    """
    Calculates the difference between STD_RT_OFF in ozon_df for each unique Sample_ID
    with the reference STD_RT_OFF from fame_std and applies the same difference to all rows for that Sample_ID.
    Creates a new column 'Adjusted_RT_FAME' which is the sum of 'Retention_Time' and 'STD_RT_Dif_FAME'.
    
    Assumes fame_std contains only one reference STD_RT_OFF value.
    """
    # Get the reference STD_RT_OFF value from fame_std
    reference_std_rt_off = fame_std['STD_RT_OFF'].iloc[0]
    print(f"Reference STD_RT_OFF: {reference_std_rt_off}")

    # Create a dictionary to store STD_RT_Dif_FAME for each unique Sample_ID
    std_rt_diff_dict = {}
    
    # Calculate the difference for each unique Sample_ID in ozon_df
    for sample_id in ozon_df['Sample_ID'].unique():
        sample_std_rt_off = ozon_df.loc[ozon_df['Sample_ID'] == sample_id, 'STD_RT_OFF'].iloc[0]
        std_rt_diff_dict[sample_id] = sample_std_rt_off - reference_std_rt_off
        print(f"Sample_ID: {sample_id}, STD_RT_OFF: {sample_std_rt_off}, "
              f"STD_RT_Dif_FAME: {std_rt_diff_dict[sample_id]}")
    
    # Apply the calculated difference to all rows based on Sample_ID
    ozon_df['STD_RT_Dif_FAME'] = ozon_df['Sample_ID'].map(std_rt_diff_dict)
    print("STD_RT_Dif_FAME column populated with RT differences per Sample_ID.")
    
    # Calculate Adjusted_RT_FAME as Retention_Time + STD_RT_Dif_FAME
    ozon_df['Adjusted_RT_FAME'] = ozon_df['Retention_Time'] + ozon_df['STD_RT_Dif_FAME']
    print("Adjusted_RT_FAME column created with adjusted retention times.")

# Example usage:
# STD_RT_Adjustment(ozon_df, fame_std)


In [ ]:
fame_std = pd.read_parquet('Projects/STD/off_possible/FAME_off_possible_top2.parquet')
fame_std.to_excel('Projects/AMP/results/nov07/FAME.xlsx')
# fame_std = fame_std[fame_std['Species'].str.contains('16:1', case=False, na=False)]
fame_std

In [ ]:
STD_RT_Adjustment(df_analysis, fame_std)
df_analysis.to_parquet('Projects/AMP/notpossible/rt_adjustment_6/df_rt_adjustment_6_hippo_WT_m1_FAD245_OFF.parquet')
df_analysis

# 6 - FILTER BY FAME UPDATEING Nov6

# nov7 testing for all FA not just 16:1 (WORKING WHEN ONLY 1 FAME value like only 16:1 but no others)

In [ ]:
# import pandas as pd
# import os
# import re

# def filter_OzOFF_by_fame_std(input_dir, fame_std, output_dir, fame_rt_window=0.5):
#     """
#     Filters parquet files in the input directory based on fame_std retention times and species,
#     using Adjusted_RT_FAME from the input files to compare against fame_std Retention_Time.
#     Updates Species and Lipid to only contain the matched species (including trailing details) and saves
#     the filtered result as parquet files in the output directory. Logs and removed lipids CSVs are saved
#     in a log directory inside the output directory.
#     """
    
#     print("Starting filter_OzOFF_by_fame_std function")
#     print(f"Input directory: {input_dir}")
#     print(f"Output directory: {output_dir}")
#     print(f"Retention time window: {fame_rt_window}")
#     print(f"Fame standard DataFrame columns: {fame_std.columns.tolist()}")

#     # Ensure the output directory exists
#     os.makedirs(output_dir, exist_ok=True)

#     # Create a log subdirectory inside the output directory
#     log_dir = os.path.join(output_dir, 'log')
#     os.makedirs(log_dir, exist_ok=True)

#     # Check if input directory has parquet files
#     input_files = [f for f in os.listdir(input_dir) if f.endswith(".parquet")]
#     print(f"Number of parquet files found in input directory: {len(input_files)}")
#     if not input_files:
#         print("No parquet files found in input directory. Exiting function.")
#         return

#     # Iterate over all parquet files in the input directory
#     for parquet_file in input_files:
#         print(f"Processing file: {parquet_file}")

#         # Read the parquet file
#         file_path = os.path.join(input_dir, parquet_file)
#         filtered_ozON = pd.read_parquet(file_path)

#         # Add Lipid_Possible column as a copy of Lipid
#         filtered_ozON['Lipid_Possible'] = filtered_ozON['Lipid']
#         print("Lipid_Possible column created")

#         # Create a new DataFrame to store the filtered ozON data
#         filtered_ozON_result = filtered_ozON.copy()

#         # Prepare lists for logging
#         drop_indices = []
#         removed_lipids = []

#         # Get sample name from Sample column
#         sample_name = filtered_ozON_result['Sample'].unique()[0]
#         log_filename = f"{sample_name}_fame_filter_log_debug.txt"
#         log_file_path = os.path.join(log_dir, log_filename)

#         # CSV filename for removed lipids
#         csv_filename = f"{sample_name}_fame_filter_removed_lipids_debug.csv"
#         csv_file_path = os.path.join(log_dir, csv_filename)

#         # Iterate through each entry in fame_std to filter ozON entries
#         for _, fame_entry in fame_std.iterrows():
#             fame_std_rt = fame_entry['Retention_Time']
#             fame_std_species = fame_entry['Species']
#             print(f"Processing fame_std entry: Retention_Time={fame_std_rt}, Species={fame_std_species}")

#             # Define bounds for Adjusted_RT_FAME comparison
#             rt_lower_bound = fame_std_rt - fame_rt_window
#             rt_upper_bound = fame_std_rt + fame_rt_window
#             print(f"Adjusted RT bounds: {rt_lower_bound} to {rt_upper_bound}")

#             # Updated regex pattern to match the entire lipid entry with "FA()" and trailing details
#             pattern = rf'\bFA\({re.escape(fame_std_species)}\)(?:_[^|]*)?'

#             # Filter entries within the defined Adjusted_RT_FAME window and matching Species
#             for index, row in filtered_ozON_result.iterrows():
#                 adjusted_rt = row['Adjusted_RT_FAME']
#                 species_list = row['Species'].split('|')  # Split species by '|'

#                 print(f"Row {index}: Adjusted_RT_FAME={adjusted_rt}, Species={row['Species']}")
#                 print(f"Species list after split: {species_list}")

#                 # Check if fame_std_species is in the species_list and adjusted_rt is within bounds
#                 if fame_std_species in species_list and rt_lower_bound <= adjusted_rt <= rt_upper_bound:
#                     print(f"Match found for row {index}: Adjusted_RT_FAME within bounds and Species match")
                    
#                     # Apply regex to capture the full matched lipid entry with details
#                     matched_lipid = '|'.join(re.findall(pattern, row['Lipid']))
                    
#                     # Debug statement to check the matched_lipid content before assignment
#                     print(f"Original Lipid: {row['Lipid']}")
#                     print(f"Pattern used: {pattern}")
#                     print(f"Matched Lipid after regex: {matched_lipid}")
                    
#                     # Update Species and Lipid columns to contain only the matched fame_std_species with details
#                     filtered_ozON_result.at[index, 'Species'] = fame_std_species
#                     filtered_ozON_result.at[index, 'Lipid'] = matched_lipid
                    
#                     # Confirm the updated values
#                     print(f"Updated Species for row {index} to: {fame_std_species}")
#                     print(f"Updated Lipid for row {index} to: {matched_lipid}")
#                 else:
#                     # If it doesn't match, log the row for removal
#                     rt_diff = min(abs(adjusted_rt - rt_lower_bound), abs(adjusted_rt - rt_upper_bound))
#                     print(f"No match for row {index}: RT difference={rt_diff}, Species mismatch or RT out of bounds")
#                     drop_indices.append(index)
#                     removed_lipids.append({
#                         'Lipid': row['Lipid'],
#                         'Adjusted_RT_FAME': adjusted_rt,
#                         'Ground_Truth_RT': fame_std_rt,
#                         'Species': row['Species'],
#                         'Ground_Truth_Species': fame_std_species,
#                         'RT_Difference': rt_diff
#                     })

#         # Drop all collected indices
#         filtered_ozON_result = filtered_ozON_result.drop(drop_indices).reset_index(drop=True)
#         print(f"Filtered rows in {sample_name}: {len(drop_indices)} removed")

#         # Save filtered DataFrame and removed lipids log to CSV
#         filtered_ozON_result.to_parquet(os.path.join(output_dir, f"{sample_name}_filtered.parquet"))
#         pd.DataFrame(removed_lipids).to_csv(csv_file_path, index=False)

#         # Write log entries to a text file
#         with open(log_file_path, 'w') as log_file:
#             for entry in removed_lipids:
#                 log_file.write(str(entry) + '\n')
#             print(f"Log for removed lipids saved at: {log_file_path}")

#     print("filter_OzOFF_by_fame_std function completed.")


# Try to make work for more values

In [ ]:
# import pandas as pd
# import os
# import re

# def filter_OzOFF_by_fame_std(input_dir, fame_std, output_dir, fame_rt_window=0.5):
#     print("Starting filter_OzOFF_by_fame_std function")
#     print(f"Input directory: {input_dir}")
#     print(f"Output directory: {output_dir}")
#     print(f"Retention time window: {fame_rt_window}")
#     print(f"Fame standard DataFrame columns: {fame_std.columns.tolist()}")

#     os.makedirs(output_dir, exist_ok=True)
#     log_dir = os.path.join(output_dir, 'log')
#     os.makedirs(log_dir, exist_ok=True)

#     input_files = [f for f in os.listdir(input_dir) if f.endswith(".parquet")]
#     print(f"Number of parquet files found in input directory: {len(input_files)}")
#     if not input_files:
#         print("No parquet files found in input directory. Exiting function.")
#         return

#     for parquet_file in input_files:
#         print(f"Processing file: {parquet_file}")
#         file_path = os.path.join(input_dir, parquet_file)
#         filtered_ozON = pd.read_parquet(file_path)
#         filtered_ozON['Lipid_Possible'] = filtered_ozON['Lipid']
#         print("Lipid_Possible column created")

#         # Final result DataFrame to accumulate results from each fame_std entry
#         final_filtered_result = pd.DataFrame()

#         sample_name = filtered_ozON['Sample'].unique()[0]
#         log_filename = f"{sample_name}_fame_filter_log_debug.txt"
#         log_file_path = os.path.join(log_dir, log_filename)
#         csv_filename = f"{sample_name}_fame_filter_removed_lipids_debug.csv"
#         csv_file_path = os.path.join(log_dir, csv_filename)

#         # Iterate through each entry in fame_std independently
#         for _, fame_entry in fame_std.iterrows():
#             fame_std_rt = fame_entry['Retention_Time']
#             fame_std_species = fame_entry['Species']
#             print(f"Processing fame_std entry: Retention_Time={fame_std_rt}, Species={fame_std_species}")

#             rt_lower_bound = fame_std_rt - fame_rt_window
#             rt_upper_bound = fame_std_rt + fame_rt_window
#             print(f"Adjusted RT bounds: {rt_lower_bound} to {rt_upper_bound}")

#             pattern = rf'\bFA\({re.escape(fame_std_species)}\)(?:_[^|]*)?'

#             # Create a copy of filtered_ozON to apply the filter for the current fame_std entry
#             filtered_result = filtered_ozON.copy()
#             filtered_result['Matched'] = False

#             for index, row in filtered_result.iterrows():
#                 adjusted_rt = row['Adjusted_RT_FAME']
#                 species_list = row['Species'].split('|')
#                 print(f"Row {index}: Adjusted_RT_FAME={adjusted_rt}, Species={row['Species']}")

#                 if fame_std_species in species_list and rt_lower_bound <= adjusted_rt <= rt_upper_bound:
#                     matched_lipid = '|'.join(re.findall(pattern, row['Lipid']))
#                     print(f"Original Lipid: {row['Lipid']}")
#                     print(f"Pattern used: {pattern}")
#                     print(f"Matched Lipid after regex: {matched_lipid}")

#                     filtered_result.at[index, 'Species'] = fame_std_species
#                     filtered_result.at[index, 'Lipid'] = matched_lipid
#                     filtered_result.at[index, 'Matched'] = True
#                     print(f"Updated Species for row {index} to: {fame_std_species}")
#                     print(f"Updated Lipid for row {index} to: {matched_lipid}")

#             # Append only the matched rows to the final result DataFrame
#             final_filtered_result = pd.concat([final_filtered_result, filtered_result[filtered_result['Matched']]])

#         # Remove 'Matched' column and reset index for the final filtered DataFrame
#         final_filtered_result = final_filtered_result.drop(columns=['Matched']).reset_index(drop=True)

#         # Save filtered DataFrame and logs
#         final_filtered_result.to_parquet(os.path.join(output_dir, f"{sample_name}_filtered.parquet"))
#         print(f"Filtered data saved to {sample_name}_filtered.parquet")

#     print("filter_OzOFF_by_fame_std function completed.")


In [ ]:
# #from core.python.fame_filter_AMP_7 import filter_ozon_by_fame_std
# import pandas as pd

# # Define input and output directories
# #input_dir = 'Projects/AMP/analysis/OFF/notpossible/'
# input_dir = 'Projects/AMP/notpossible/rt_adjustment_6/'
# output_dir = 'Projects/AMP/fame_filter_7/notpossible_nov6/'


# # Call the function with custom fame_rt_window and parent_ion_tolerance
# filter_OzOFF_by_fame_std(input_dir, fame_std, output_dir, fame_rt_window=0.5)


In [ ]:
import pandas as pd
#step7 = pd.read_parquet('Projects/AMP/fame_filter_7/notpossible_nov6/hippo_WT_m1_FAD245_filtered.parquet')
step7 = pd.read_parquet('Projects/AMP/analysis/OFF/notpossible/df_analysis_5_hippo_WT_m1_FAD245_OFF.parquet')
# step7.to_excel('Projects/AMP/results/nov07/hippo_WT_m1_FAD245_notpossibleOzOFF_nov07.xlsx')

#print unique Lipid values
print(step7['Species'].unique())
step7

In [ ]:
# !sbatch core/backend/not_possible/FAME_filter_6_notpossible.sh

In [ ]:

# import pandas as pd
# # from core.python.not_possible.FAME_filter_6_notpossible import filter_OzOFF_by_fame_std

# # Define input and output directories
# #input_dir = 'Projects/AMP/analysis/OFF/notpossible/'
# input_dir = 'Projects/AMP/notpossible/rt_adjustment_6/'
# output_dir = 'Projects/AMP/fame_filter_7/notpossible_nov6/'
# fame_std = pd.read_parquet('Projects/STD/off_possible/FAME_off_possible_top2.parquet')


# # Call the function with custom fame_rt_window and parent_ion_tolerance
# filter_OzOFF_by_fame_std(input_dir, fame_std, output_dir, fame_rt_window=0.5)


# OzOFF notpossbile - Remove OzON not possible  | LOAD OZON HERE

In [ ]:
import pandas as pd
#ozon_test = pd.read_parquet('Projects/AMP/fame_filter_7/hippo_WT_m1_FAD245_fame_filter_7.parquet')
ozon_test = pd.read_parquet('Projects/AMP/isomer_filter_6/hippo_WT_m1_FAD245_isomer_filtered_6.parquet')

#filtered_lipids = ozon_test[ozon_test['Lipid'].str.contains('18:1', case=False, na=False)]
filtered_lipids = ozon_test[ozon_test['Lipid'].str.contains('18:1|16:1', case=False, na=False)]

sorted_filtered_lipids_OZON = filtered_lipids.sort_values(by='OzESI_Intensity', ascending=False)
print(sorted_filtered_lipids_OZON['Species'].unique())
sorted_filtered_lipids_OZON = sorted_filtered_lipids_OZON.sort_values(by='Lipid', ascending=True)
sorted_filtered_lipids_OZON


# ozon_test

# OZ OFF NOT POSSIBLE 1 FILE AT A TIME

In [ ]:
import pandas as pd

def match_lipids_with_adjusted_rt_to_rt(ozon_test, sorted_notpossible_lipids, rt_window=0.5):
    # Create new columns in the ON data to store matched information from OFF
    ozon_test['Matched_Lipid_OFF'] = None
    ozon_test['Intensity_OFF'] = None
    ozon_test['Retention_Time_OFF'] = None

    # Iterate through each row in the ON data
    for index_on, row_on in ozon_test.iterrows():
        on_lipid = row_on['Lipid'].strip()  # Trim spaces for accurate matching
        on_adjusted_rt = row_on['Adjusted_RT']  # Adjusted RT of the ON lipid
        match_found = False

        # Iterate through each row in the OFF data
        for index_off, row_off in sorted_notpossible_lipids.iterrows():
            # Split the OFF Lipid values by '|' and trim each part
            off_lipids = [lipid.strip() for lipid in row_off['Lipid'].split('|')]
            off_retention_time = row_off['Retention_Time']  # Retention time of the OFF lipid

            # Check if the ON lipid matches any of the possible OFF lipids
            # and if the Adjusted_RT is within the specified window of the Retention_Time
            if any(on_lipid.lower() == off_lipid.lower() for off_lipid in off_lipids) and abs(on_adjusted_rt - off_retention_time) <= rt_window:
                # If a match is found, update the 'Matched_Lipid_OFF', 'Intensity_OFF', and 'Retention_Time_OFF' columns in the ON DataFrame
                ozon_test.at[index_on, 'Matched_Lipid_OFF'] = row_off['Lipid']
                ozon_test.at[index_on, 'Intensity_OFF'] = row_off['OzESI_Intensity']
                ozon_test.at[index_on, 'Retention_Time_OFF'] = off_retention_time
                match_found = True
                break  # Stop searching for this ON lipid once a match is found

        # If no match is found, log or handle this scenario
        if not match_found:
            print(f"No match found for ON lipid: {on_lipid} within Adjusted_RT window {on_adjusted_rt} ± {rt_window}")

    # Filter the ON data to separate matched and unmatched lipids
    matched_lipids_df = ozon_test.dropna(subset=['Matched_Lipid_OFF']).copy()
    unmatched_lipids_df = ozon_test[ozon_test['Matched_Lipid_OFF'].isna()].copy()

    return matched_lipids_df, unmatched_lipids_df

# Example usage
# Assuming sorted_notpossible_lipids and ozon_test are the input DataFrames
# RUN WITH A SMALL SECTION OF LIPIDS FROM OZ ON
matched_lipids_df, unmatched_lipids_df = match_lipids_with_adjusted_rt_to_rt(sorted_filtered_lipids_OZON, step7, rt_window=0.05)

# # RUN WITH ALL LIPIDS FROM OZON
# matched_lipids_df, unmatched_lipids_df = match_lipids_with_adjusted_rt_to_rt(ozon_test, step7, rt_window=0.1)


# # Print values of Matched_Lipid_OFF in row 1 if exists
# if not matched_lipids_df.empty:
#     print(matched_lipids_df['Matched_Lipid_OFF'].iloc[1])

# Display or save the results
matched_lipids_df
unmatched_lipids_df
# matched_lipids_df.to_csv('matched_lipids.csv', index=False)
# unmatched_lipids_df.to_csv('unmatched_lipids.csv', index=False)


In [ ]:
matched_lipids_df

# make ozon vs ozff not possible into a dir function ALL FILES AT ONCE

In [ ]:
import pandas as pd
import os
import glob

def list_files_in_dirs(dir1, dir2, extension='*.parquet'):
    """
    Lists file names from two directories and creates keys for matching.

    Parameters:
    dir1 (str): Path to the first directory.
    dir2 (str): Path to the second directory.
    extension (str): File extension to match (default: '*.parquet').

    Returns:
    DataFrame: DataFrame containing file names from both directories and their keys.
    """
    # Get list of files in both directories
    files1 = glob.glob(os.path.join(dir1, extension))
    files2 = glob.glob(os.path.join(dir2, extension))

    # Debugging: Print file lists
    print(f"Files in {dir1}: {files1}")
    print(f"Files in {dir2}: {files2}")

    # Create separate DataFrames for each directory
    df1 = pd.DataFrame({
        'File': [os.path.basename(f) for f in files1],
        'Key': ["_".join(os.path.basename(f).split('_')[:4]).replace('.parquet', '') for f in files1]
    })
    df2 = pd.DataFrame({
        'File': [os.path.basename(f) for f in files2],
        'Key': ["_".join(os.path.basename(f).split('_')[3:7]).replace('.parquet', '') for f in files2]
    })

    # Debugging: Print DataFrames before processing
    print(f"Initial DataFrame 1:\n{df1}")
    print(f"Initial DataFrame 2:\n{df2}")

    # Remove '5_' prefix from keys if present
    df1['Key'] = df1['Key'].str.replace('^5_', '', regex=True)
    df2['Key'] = df2['Key'].str.replace('^5_', '', regex=True)

    # Debugging: Print DataFrames after processing keys
    print(f"Processed DataFrame 1:\n{df1}")
    print(f"Processed DataFrame 2:\n{df2}")

    # Merge the DataFrames on the Key column
    merged_df = pd.merge(
        df1, 
        df2, 
        on='Key', 
        how='inner',  # Only keep pairs with matching keys
        suffixes=('_OzON', '_OzOFF')
    )

    # Debugging: Print merged DataFrame
    print(f"Merged DataFrame:\n{merged_df}")

    return merged_df


def match_lipids_with_adjusted_rt_to_rt_from_dirs(dir1, dir2, output_dir, rt_window=0.5):
    """
    Matches lipids between files in two directories using retention time and keys.

    Parameters:
    dir1 (str): Directory containing OzON files.
    dir2 (str): Directory containing OzOFF files.
    output_dir (str): Directory to save matched and unmatched results.
    rt_window (float): Retention time window for matching (default: 0.5).
    """
    # Get matched file pairs based on keys
    file_pairs = list_files_in_dirs(dir1, dir2)

    # Iterate through each matched pair of files
    for _, row in file_pairs.iterrows():
        ozon_file = os.path.join(dir1, row['File_OzON'])
        ozoff_file = os.path.join(dir2, row['File_OzOFF'])

        # Load the data from the matched files
        ozon_test = pd.read_parquet(ozon_file)
        sorted_notpossible_lipids = pd.read_parquet(ozoff_file)

        # Debugging: Print the first few rows of both DataFrames
        print(f"OzON DataFrame for {row['File_OzON']}:\n{ozon_test.head()}")
        print(f"OzOFF DataFrame for {row['File_OzOFF']}:\n{sorted_notpossible_lipids.head()}")

        # Create new columns in the ON data to store matched information from OFF
        ozon_test['Matched_Lipid_OFF'] = None
        ozon_test['Intensity_OFF'] = None
        ozon_test['Retention_Time_OFF'] = None

        # Perform lipid matching
        for index_on, row_on in ozon_test.iterrows():
            on_lipid = row_on['Lipid'].strip()  # Trim spaces for accurate matching
            on_adjusted_rt = row_on['Adjusted_RT']
            match_found = False

            for _, row_off in sorted_notpossible_lipids.iterrows():
                off_lipids = [lipid.strip() for lipid in row_off['Lipid'].split('|')]
                off_retention_time = row_off['Retention_Time']

                # Debugging: Print lipid comparison details
                print(f"Comparing ON lipid {on_lipid} (RT: {on_adjusted_rt}) with OFF lipids {off_lipids} (RT: {off_retention_time})")

                if any(on_lipid.lower() == off_lipid.lower() for off_lipid in off_lipids) and abs(on_adjusted_rt - off_retention_time) <= rt_window:
                    ozon_test.at[index_on, 'Matched_Lipid_OFF'] = row_off['Lipid']
                    ozon_test.at[index_on, 'Intensity_OFF'] = row_off['OzESI_Intensity']
                    ozon_test.at[index_on, 'Retention_Time_OFF'] = off_retention_time
                    match_found = True
                    break

            if not match_found:
                print(f"No match found for ON lipid {on_lipid} with Adjusted RT {on_adjusted_rt}")

        # Separate matched and unmatched lipids
        matched_lipids_df = ozon_test.dropna(subset=['Matched_Lipid_OFF']).copy()
        unmatched_lipids_df = ozon_test[ozon_test['Matched_Lipid_OFF'].isna()].copy()

        # Save results
        os.makedirs(output_dir, exist_ok=True)  # Ensure output directory exists
        matched_file = os.path.join(output_dir, f"matched_{row['Key']}.csv")
        unmatched_file = os.path.join(output_dir, f"unmatched_{row['Key']}.csv")

        matched_lipids_df.to_csv(matched_file, index=False)
        unmatched_lipids_df.to_csv(unmatched_file, index=False)

        print(f"Processed files: {row['File_OzON']} and {row['File_OzOFF']}")
        print(f"Saved matched lipids to {matched_file}")
        print(f"Saved unmatched lipids to {unmatched_file}")


# Example usage
ozon_dir = 'Projects/AMP/isomer_filter_6/'
ozoff_dir = 'Projects/AMP/analysis/OFF/notpossible/'
output_dir = 'Projects/AMP/notpossible_9/'

match_lipids_with_adjusted_rt_to_rt_from_dirs(ozon_dir, ozoff_dir, output_dir, rt_window=0.05)


# SH WORKING FOR ALL FILES AT ONCE

In [ ]:
import glob
import pandas as pd

# Define directories
off_dir = 'Projects/AMP/analysis/OFF/notpossible/'
on_dir = 'Projects/AMP/isomer_filter_6/'

# Count files in the OFF directory
num_files_off = len(glob.glob(f"{off_dir}*.parquet"))
num_files_on = len(glob.glob(f"{on_dir}*.parquet"))
print(f"Number of files in OFF directory: {num_files_off}")
print(f"Number of files in ON directory: {num_files_on}")



In [ ]:
!sbatch core/backend/not_possible/notpossible_9.sh 0.05

# POSSIBLE

In [ ]:
# Sort unmatched_lipids_df by 'Lipid' column in ascending order
unmatched_lipids_df = unmatched_lipids_df.sort_values(by='Lipid', ascending=True)

# If you want to sort in descending order, set ascending=False
# unmatched_lipids_df = unmatched_lipids_df.sort_values(by='Lipid', ascending=False)
#print columns
print(unmatched_lipids_df.columns)

unmatched_lipids_df

In [ ]:
import pandas as pd

def literature_removal(df):
    # Filter rows with 'n-2' or 'n-3' in the Lipid column
    filtered_df = df[df['Lipid'].str.contains('n-2|n-3', na=False)].copy()
    
    # Remove rows with 'n-2' or 'n-3' from the original DataFrame
    original_df = df[~df['Lipid'].str.contains('n-2|n-3', na=False)].copy()
    
    return original_df, filtered_df

def filter_highest_intensity(df):
    """
    Filters the DataFrame to keep only the row with the highest OzESI_Intensity
    for each unique combination of Lipid and OzOFF_Isomer.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A filtered DataFrame.
    """
    # Sort the DataFrame by Lipid, OzOFF_Isomer, and OzESI_Intensity in descending order
    sorted_df = df.sort_values(by=['Lipid', 'OzOFF_Isomer', 'OzESI_Intensity'], ascending=[True, True, False])
    
    # Drop duplicates, keeping the first (highest intensity) for each Lipid and OzOFF_Isomer combination
    filtered_df = sorted_df.drop_duplicates(subset=['Lipid', 'OzOFF_Isomer'], keep='first')
    
    return filtered_df
def duplicate_removal(df):
    """
    Removes duplicate rows based on Lipid and OzESI_Intensity columns.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with duplicates removed based on specified columns.
    """
    return df.drop_duplicates(subset=['Lipid', 'OzESI_Intensity'])



# Example usage
# original_df, filtered_df = literature_removal(unmatched_lipids_df)

# Example usage
original_df, n_2_df = literature_removal(unmatched_lipids_df)
original_df

# Step 2: Remove duplicates based on Lipid and OzESI_Intensity
original_df = duplicate_removal(original_df)
# Example usage
original_df = filter_highest_intensity(original_df)
original_df = original_df.drop_duplicates()
original_df.to_excel('Projects/AMP/notpossible/manual_validation/hippo_WT_m1_FAD245_notpossible_nov11.xlsx')
original_df



In [ ]:
# Remove rows with identical values in Lipid, Retention_Time, and OzESI_Intensity columns
original_df = original_df.drop_duplicates(subset=['Lipid', 'Retention_Time', 'OzESI_Intensity'])
#original_df.to_excel('Projects/AMP/results/nov07/hippo_WT_m1_FAD245_unmatched_lipids.xlsx', index=False)
original_df


In [ ]:
# Filter original_df for rows where Species is '16:1'
species_16_1_df = original_df[original_df['Species'] == '18:1']

# Display the filtered DataFrame
species_16_1_df


# UNLIKELY 

In [ ]:
matched_lipids_df

In [ ]:
import pandas as pd
step9 = pd.read_csv('Projects/AMP/notpossible_9/unmatched_hippo_WT_m1_FAD245.csv')
# step9 = pd.read_csv('Projects/AMP/notpossible_9/matched_hippo_WT_m1_FAD245.csv')
# Example usage
step9, n_2_df = literature_removal(step9)
step9
step9 = step9[step9['Lipid'].str.contains('18:1|16:1', case=False, na=False)]
# Step 2: Remove duplicates based on Lipid and OzESI_Intensity
step9 = duplicate_removal(step9)
# Example usage
step9 = filter_highest_intensity(step9)
step9 = step9.drop_duplicates()

step9

# review 9 possible script df

In [ ]:
import pandas as pd
step9 = pd.read_csv('Projects/AMP/notpossible_9/unmatched_hippo_WT_m1_FAD245.csv')
# step9 = pd.read_csv('Projects/AMP/notpossible_9/matched_hippo_WT_m1_FAD245.csv')
# Example usage
step9, n_2_df = literature_removal(step9)
step9
step9 = step9[step9['Lipid'].str.contains('18:1|16:1', case=False, na=False)]
# Step 2: Remove duplicates based on Lipid and OzESI_Intensity
step9 = duplicate_removal(step9)
# Example usage
step9 = filter_highest_intensity(step9)
step9 = step9.drop_duplicates()
step9

In [ ]:
import pandas as pd

def literature_removal(df):
    # Filter rows with 'n-2' or 'n-3' in the Lipid column
    filtered_df = df[df['Lipid'].str.contains('n-2|n-3', na=False)].copy()
    
    # Remove rows with 'n-2' or 'n-3' from the original DataFrame
    original_df = df[~df['Lipid'].str.contains('n-2|n-3', na=False)].copy()
    
    return original_df, filtered_df

def filter_highest_intensity(df):
    """
    Filters the DataFrame to keep only the row with the highest OzESI_Intensity
    for each unique combination of Lipid and OzOFF_Isomer.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A filtered DataFrame.
    """
    # Sort the DataFrame by Lipid, OzOFF_Isomer, and OzESI_Intensity in descending order
    sorted_df = df.sort_values(by=['Lipid', 'OzOFF_Isomer', 'OzESI_Intensity'], ascending=[True, True, False])
    
    # Drop duplicates, keeping the first (highest intensity) for each Lipid and OzOFF_Isomer combination
    filtered_df = sorted_df.drop_duplicates(subset=['Lipid', 'OzOFF_Isomer'], keep='first')
    
    return filtered_df
def duplicate_removal(df):
    """
    Removes duplicate rows based on Lipid and OzESI_Intensity columns.

    Args:
        df (pd.DataFrame): The input DataFrame.

    Returns:
        pd.DataFrame: A DataFrame with duplicates removed based on specified columns.
    """
    return df.drop_duplicates(subset=['Lipid', 'OzESI_Intensity'])

In [ ]:
from utils import literature_removal, filter_highest_intensity, duplicate_removal
import pandas as pd
step9a = pd.read_csv('Projects/AMP/notpossible_9/unmatched_hippo_WT_m1_FAD245.csv')
# Example usage
step9a, n_2_df = literature_removal(step9a)
step9a = filter_highest_intensity(step9a)
step9a = step9a.drop_duplicates(subset=['Lipid', 'OzESI_Intensity'])
step9a = step9a[step9a['Lipid'].str.contains('18:1|16:1', case=False, na=False)]
step9a


In [ ]:
from utils import literature_removal, filter_highest_intensity, duplicate_removal
import pandas as pd
master = pd.read_parquet('Projects/AMP/notpossible_9/master_unmatched.parquet')
master, n_2_df = literature_removal(master)
master = filter_highest_intensity(master)
master = master.drop_duplicates(subset=['Lipid', 'OzESI_Intensity', 'Sample'])
master = master[master['Lipid'].str.contains('18:1|16:1', case=False, na=False)]
# sort by Sample and Lipid
master = master.sort_values(by=['Sample', 'Species','n_value'], ascending=True)
master.to_excel('Projects/AMP/notpossible_9/master_unmatched_nov12.xlsx')
master

# all values nov17

In [6]:
from utils import literature_removal, filter_highest_intensity, duplicate_removal
import pandas as pd
master = pd.read_parquet('Projects/AMP/notpossible_9/master_unmatched.parquet')
master = master[master['Sample'].str.contains('hippo_WT_m1_FAD245', case=False, na=False)]
master, n_2_df = literature_removal(master)
master = filter_highest_intensity(master)
master = master.drop_duplicates(subset=['Lipid', 'OzESI_Intensity', 'Sample'])
#save as excel
master.to_excel('Projects/AMP/notpossible_9/hippo_WT_m1_FAD245_ALL_nov17.xlsx')
master

,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer,Matched_Lipid_OFF,Intensity_OFF,Retention_Time_OFF
6711,FA(12:2)_<B>_n-5,3.447150,577.0,5,18,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,309.1 -> 183.0,hippo_WT_m1_FAD245,309.1,183.0,...,WT,FAD245,m1,0.000223,5,5,1.0,None,None,None
6710,FA(12:2)_<F>_n-4,3.549900,972.0,8,22,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m1_FAD245,325.2,183.0,...,WT,FAD245,m1,0.000249,4,4,1.0,None,None,None
6713,FA(13:1)_<>_n-4,4.776933,631.0,10,32,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,339.3 -> 183.0,hippo_WT_m1_FAD245,339.3,183.0,...,WT,FAD245,m1,0.000354,4,4,2.0,None,None,None
6716,FA(13:1)_<>_n-5,3.549900,972.0,8,33,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m1_FAD245,325.2,183.0,...,WT,FAD245,m1,0.000249,5,5,1.0,None,None,None
6714,FA(13:1)_<>_n-5,4.635217,953.0,8,33,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m1_FAD245,325.2,183.0,...,WT,FAD245,m1,0.001538,5,5,2.0,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6932,FA(22:6)_<FFFFF>_n-16,6.160617,604.0,4,415,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,297.2 -> 183.0,hippo_WT_m1_FAD245,297.2,183.0,...,WT,FAD245,m1,0.000325,16,16,1.0,None,None,None
6917,FA(22:6)_<FFFFF>_n-6,7.550300,1770.0,31,419,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,437.4 -> 183.0,hippo_WT_m1_FAD245,437.4,183.0,...,WT,FAD245,m1,0.002737,6,6,2.0,None,None,None
6928,FA(22:6)_<FFFFF>_n-7,6.181567,657.0,28,420,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,423.3 -> 183.0,hippo_WT_m1_FAD245,423.3,183.0,...,WT,FAD245,m1,0.000370,7,7,1.0,None,None,None
6919,FA(22:6)_<FFFFF>_n-7,7.576950,5026.0,28,420,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,423.3 -> 183.0,hippo_WT_m1_FAD245,423.3,183.0,...,WT,FAD245,m1,0.001905,7,7,2.0,None,None,None


In [1]:
import pandas as pd
master = pd.read_excel('Projects/AMP/notpossible_9/master_unmatched_nov12.xlsx')
master

,Unnamed: 0,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,...,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer,Matched_Lipid_OFF,Intensity_OFF,Retention_Time_OFF
0,4427,FA(16:1)_<>_n-7,7.089683,510,10,59,11192023_5xFAD-m1-cereb-FAD231_AMP_2x_0.05uMd2...,339.3 -> 183.0,cereb_5xFAD_m1_FAD231,339.3,...,5xFAD,FAD231,m1,0.000240,7,7,2,NaN,NaN,NaN
1,4432,FA(16:1)_<>_n-8,6.909200,831,8,60,11192023_5xFAD-m1-cereb-FAD231_AMP_2x_0.05uMd2...,325.2 -> 183.0,cereb_5xFAD_m1_FAD231,325.2,...,5xFAD,FAD231,m1,0.000803,8,8,1,NaN,NaN,NaN
2,4428,FA(16:1)_<>_n-10,7.077967,3683,4,49,11192023_5xFAD-m1-cereb-FAD231_AMP_2x_0.05uMd2...,297.2 -> 183.0,cereb_5xFAD_m1_FAD231,297.2,...,5xFAD,FAD231,m1,0.003720,10,10,2,NaN,NaN,NaN
3,4459,FA(18:1)_<>_n-5,8.624967,596,21,106,11192023_5xFAD-m1-cereb-FAD231_AMP_2x_0.05uMd2...,395.3 -> 183.0,cereb_5xFAD_m1_FAD231,395.3,...,5xFAD,FAD231,m1,0.000647,5,5,1,NaN,NaN,NaN
4,4465,FA(18:1)_<>_n-5,8.896300,1140,21,106,11192023_5xFAD-m1-cereb-FAD231_AMP_2x_0.05uMd2...,395.3 -> 183.0,cereb_5xFAD_m1_FAD231,395.3,...,5xFAD,FAD231,m1,0.001714,5,5,2,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
803,2491,FA(18:1)_<>_n-9,8.665950,56484,10,151,11162023_WT-m5-hippo-FAD263_AMP_2x_0.05uMd216-...,339.3 -> 183.0,hippo_WT_m5_FAD263,339.3,...,WT,FAD263,m5,0.045779,9,9,1,NaN,NaN,NaN
804,2500,FA(18:1)_<>_n-9,8.976033,11756,10,151,11162023_WT-m5-hippo-FAD263_AMP_2x_0.05uMd216-...,339.3 -> 183.0,hippo_WT_m5_FAD263,339.3,...,WT,FAD263,m5,0.008683,9,9,2,NaN,NaN,NaN
805,2492,FA(18:1)_<>_n-10,8.743883,3138,8,137,11162023_WT-m5-hippo-FAD263_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m5_FAD263,325.2,...,WT,FAD263,m5,0.002216,10,10,1,NaN,NaN,NaN
806,2501,FA(18:1)_<>_n-10,9.002283,7672,8,137,11162023_WT-m5-hippo-FAD263_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m5_FAD263,325.2,...,WT,FAD263,m5,0.005347,10,10,2,NaN,NaN,NaN


# all values remove 1 db n values so ##:2 or greater only

In [3]:
from utils import literature_removal, filter_highest_intensity, duplicate_removal
import pandas as pd

master = pd.read_parquet('Projects/AMP/notpossible_9/master_unmatched.parquet')
# Remove lipid values where FA(##:#) where :# is 1
master = master[~master['Lipid'].str.contains(r'FA\(\d+:\s*1\)', regex=True, na=False)]
master = master[master['Sample'].str.contains('hippo_WT_m1_FAD245', case=False, na=False)]





# Optionally, save the filtered DataFrame to an Excel file
# master.to_excel('filtered_master.xlsx', index=False)

master


,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer,Matched_Lipid_OFF,Intensity_OFF,Retention_Time_OFF
6710,FA(12:2)_<F>_n-4,3.549900,972.0,8,22,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m1_FAD245,325.2,183.0,...,WT,FAD245,m1,0.000249,4,4,1.0,None,None,None
6711,FA(12:2)_<B>_n-5,3.447150,577.0,5,18,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,309.1 -> 183.0,hippo_WT_m1_FAD245,309.1,183.0,...,WT,FAD245,m1,0.000223,5,5,1.0,None,None,None
6726,FA(15:3)_<BF>_n-8,3.447150,577.0,5,69,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,309.1 -> 183.0,hippo_WT_m1_FAD245,309.1,183.0,...,WT,FAD245,m1,0.000223,8,8,1.0,None,None,None
6737,FA(16:2)_<F>_n-5,5.822467,515.0,15,109,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,367.3 -> 183.0,hippo_WT_m1_FAD245,367.3,183.0,...,WT,FAD245,m1,0.000568,5,5,1.0,None,None,None
6738,FA(16:2)_<F>_n-10,6.160617,604.0,4,101,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,297.2 -> 183.0,hippo_WT_m1_FAD245,297.2,183.0,...,WT,FAD245,m1,0.000325,10,10,2.0,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6928,FA(22:6)_<FFFFF>_n-7,6.181567,657.0,28,420,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,423.3 -> 183.0,hippo_WT_m1_FAD245,423.3,183.0,...,WT,FAD245,m1,0.000370,7,7,1.0,None,None,None
6929,FA(22:6)_<BFFFF>_n-7,6.026733,581.0,27,406,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,421.4 -> 183.0,hippo_WT_m1_FAD245,421.4,183.0,...,WT,FAD245,m1,0.000269,7,7,1.0,None,None,None
6930,FA(22:6)_<BFFFF>_n-7,6.143017,638.0,27,406,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,421.4 -> 183.0,hippo_WT_m1_FAD245,421.4,183.0,...,WT,FAD245,m1,0.000578,7,7,1.0,None,None,None
6931,FA(22:6)_<FFFFF>_n-14,6.185650,534.0,8,413,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m1_FAD245,325.2,183.0,...,WT,FAD245,m1,0.000298,14,14,1.0,None,None,None


In [4]:
master2 = filter_highest_intensity(master)
master2 = master2.drop_duplicates(subset=['Lipid', 'OzESI_Intensity', 'Sample'])
master2, n_2_df = literature_removal(master2, 2, 3,5)
master2.to_excel('Projects/AMP/notpossible_9/hippo_WT_m1_FAD245_ALL_nov17_NOn-2n-3n-5.xlsx')
master2

Constructed regex pattern: n-2|n-3|n-5


,Lipid,Retention_Time,OzESI_Intensity,group_by_ion,group_by_lipid,Sample_ID,Transition,Sample,Parent_Ion,Product_Ion,...,Genotype,Cage,Mouse,Normalized_Peak_Area,n_value,n_position,OzOFF_Isomer,Matched_Lipid_OFF,Intensity_OFF,Retention_Time_OFF
6710,FA(12:2)_<F>_n-4,3.549900,972.0,8,22,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,325.2 -> 183.0,hippo_WT_m1_FAD245,325.2,183.0,...,WT,FAD245,m1,0.000249,4,4,1.0,None,None,None
6726,FA(15:3)_<BF>_n-8,3.447150,577.0,5,69,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,309.1 -> 183.0,hippo_WT_m1_FAD245,309.1,183.0,...,WT,FAD245,m1,0.000223,8,8,1.0,None,None,None
6738,FA(16:2)_<F>_n-10,6.160617,604.0,4,101,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,297.2 -> 183.0,hippo_WT_m1_FAD245,297.2,183.0,...,WT,FAD245,m1,0.000325,10,10,2.0,None,None,None
6795,FA(18:2)_<B>_n-10,7.529567,673.0,7,143,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,323.2 -> 183.0,hippo_WT_m1_FAD245,323.2,183.0,...,WT,FAD245,m1,0.000419,10,10,1.0,None,None,None
6815,FA(18:2)_<B>_n-10,7.800900,837.0,7,143,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,323.2 -> 183.0,hippo_WT_m1_FAD245,323.2,183.0,...,WT,FAD245,m1,0.000587,10,10,2.0,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6932,FA(22:6)_<FFFFF>_n-16,6.160617,604.0,4,415,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,297.2 -> 183.0,hippo_WT_m1_FAD245,297.2,183.0,...,WT,FAD245,m1,0.000325,16,16,1.0,None,None,None
6917,FA(22:6)_<FFFFF>_n-6,7.550300,1770.0,31,419,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,437.4 -> 183.0,hippo_WT_m1_FAD245,437.4,183.0,...,WT,FAD245,m1,0.002737,6,6,2.0,None,None,None
6928,FA(22:6)_<FFFFF>_n-7,6.181567,657.0,28,420,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,423.3 -> 183.0,hippo_WT_m1_FAD245,423.3,183.0,...,WT,FAD245,m1,0.000370,7,7,1.0,None,None,None
6919,FA(22:6)_<FFFFF>_n-7,7.576950,5026.0,28,420,11162023_WT-m1-hippo-FAD245_AMP_2x_0.05uMd216-...,423.3 -> 183.0,hippo_WT_m1_FAD245,423.3,183.0,...,WT,FAD245,m1,0.001905,7,7,2.0,None,None,None


# n value matching function test

In [12]:
import pandas as pd
import numpy as np
import re
import itertools

def n_value_combine(df, retention_time_threshold=0.2):
    """
    Combines lipid n-values based on specified rules:
    - The maximum number of n-values is equal to the number after the colon in 'FA'.
    - Generates all possible combinations if n-values exceed the maximum, but only for :# > 1.
    - Adds a 'Copy' column for each combination, only for :# > 1.
    - Includes 'Parent_Ion' and 'group_by_lipid' columns in the output, aggregating all relevant values separated by '| '.
    - Rounds Retention_Time to 1 decimal place in the output.

    Parameters:
    - df (DataFrame): Input DataFrame containing 'Lipid', 'Parent_Ion', 'group_by_lipid', and 'Retention_Time' columns.
    - retention_time_threshold (float): Maximum difference in retention time to consider lipids similar.

    Returns:
    - DataFrame: New DataFrame with combined lipid values, rounded Retention Times, 'Copy' column (if applicable),
               'Parent_Ion' and 'group_by_lipid' columns with values separated by '| ', and 'total_n_values' column indicating the count of n_values per species.
    """
    # Ensure necessary columns are present in the DataFrame
    required_columns = ['Parent_Ion', 'group_by_lipid', 'Lipid', 'Retention_Time']
    missing_columns = [col for col in required_columns if col not in df.columns]
    if missing_columns:
        raise ValueError(f"Input DataFrame is missing required columns: {', '.join(missing_columns)}")

    # Convert 'Parent_Ion' and 'group_by_lipid' to string to ensure consistency during concatenation
    df['Parent_Ion'] = df['Parent_Ion'].astype(str)
    df['group_by_lipid'] = df['group_by_lipid'].astype(str)

    # Extract 'FA', 'positions', 'n_value' from the 'Lipid' column
    lipid_pattern = r'(FA\(\d+:\d+\))_<(.*?)>_n-(\d+)'
    extracted = df['Lipid'].str.extract(lipid_pattern, expand=True)
    if extracted.isnull().values.any():
        raise ValueError("Some entries in 'Lipid' column do not match the expected pattern 'FA(x:y)_<positions>_n-z'.")
    df[['FA', 'positions', 'n_value']] = extracted
    df['n_value'] = df['n_value'].astype(int)
    
    # Ensure 'Retention_Time' is numeric
    df['Retention_Time'] = pd.to_numeric(df['Retention_Time'], errors='coerce')
    if df['Retention_Time'].isnull().any():
        raise ValueError("Some 'Retention_Time' values could not be converted to float.")
    df['Retention_Time'] = df['Retention_Time'].round(1)  # Round to 1 decimal place

    # Extract the number after the colon to calculate total_positions
    total_positions_pattern = r'FA\(\d+:(\d+)\)'
    extracted_positions = df['FA'].str.extract(total_positions_pattern)
    if extracted_positions.isnull().values.any():
        raise ValueError("Some entries in 'FA' column do not match the expected pattern 'FA(x:y)'.")
    df['total_positions'] = extracted_positions[0].astype(int)
    df['expected_positions'] = df['total_positions']  # Expected positions equal to total positions

    # Sort DataFrame by 'FA', 'positions', and 'Retention_Time'
    df = df.sort_values(by=['FA', 'positions', 'Retention_Time']).reset_index(drop=True)

    # Initialize a list to hold the combined results
    combined_results = []

    # Define the separator for Parent_Ion and group_by_lipid
    separator = '| '  # Pipe followed by a space

    # Group by 'FA' to process each fatty acid separately
    for fa, fa_group in df.groupby('FA'):
        expected_positions = fa_group['expected_positions'].iloc[0]
        total_positions_number = fa_group['total_positions'].iloc[0]

        # Group by 'positions' within each FA
        for positions, group in fa_group.groupby('positions'):
            group = group.reset_index(drop=True)
            # Create clusters based on retention time proximity
            while not group.empty:
                # Take the first entry as the seed of the cluster
                seed = group.iloc[0]
                seed_retention_time = seed['Retention_Time']

                # Find all entries with similar retention time
                mask = abs(group['Retention_Time'] - seed_retention_time) <= retention_time_threshold
                cluster = group[mask]
                group = group[~mask]

                # Collect unique n_values and their retention times, Parent_Ion, and group_by_lipid
                n_values = sorted(set(cluster['n_value'].tolist()))
                n_values_times = dict(zip(cluster['n_value'], cluster['Retention_Time']))
                n_values_parent_ions = dict(zip(cluster['n_value'], cluster['Parent_Ion']))
                n_values_group_by_lipid = dict(zip(cluster['n_value'], cluster['group_by_lipid']))

                num_n_values = len(n_values)

                # Check if total_positions_number is greater than 1
                if total_positions_number > 1:
                    # If the number of n_values exceeds expected_positions, generate combinations
                    if num_n_values > expected_positions:
                        combinations_list = list(itertools.combinations(n_values, expected_positions))
                        for idx, comb in enumerate(combinations_list, 1):
                            n_values_str = ' '.join([f"n-{n}" for n in comb])
                            retention_times = [str(n_values_times[n]) for n in comb]
                            retention_times_str = ','.join(retention_times)
                            # Aggregate Parent_Ion and group_by_lipid using the defined separator
                            parent_ions = [n_values_parent_ions[n] for n in comb]
                            parent_ions_str = separator.join(parent_ions)
                            group_by_lipids = [n_values_group_by_lipid[n] for n in comb]
                            group_by_lipids_str = separator.join(group_by_lipids)

                            combined_lipid = f"{fa} {n_values_str}"

                            combined_results.append({
                                'Lipid': combined_lipid,
                                'Parent_Ion': parent_ions_str,  # Aggregated Parent_Ion values
                                'group_by_lipid': group_by_lipids_str,  # Aggregated group_by_lipid values
                                'Retention_Time': retention_times_str,
                                'Copy': f"Copy{idx}",
                                'total_n_values': num_n_values  # Count of n_values in the cluster
                            })
                    else:
                        # Number of missing positions
                        num_missing_positions = expected_positions - num_n_values
                        # Create positions marker with the correct number of 'x's
                        if num_missing_positions > 0:
                            positions_marker = '<' + 'x' * num_missing_positions + '>'
                        else:
                            positions_marker = ''

                        n_values_str = ' '.join([f"n-{n}" for n in n_values])
                        combined_lipid = f"{fa} {n_values_str} {positions_marker}".strip()
                        retention_times = [str(n_values_times[n]) for n in n_values]
                        retention_times_str = ','.join(retention_times)
                        parent_ions = [n_values_parent_ions[n] for n in n_values]
                        parent_ions_str = separator.join(parent_ions)
                        group_by_lipids = [n_values_group_by_lipid[n] for n in n_values]
                        group_by_lipids_str = separator.join(group_by_lipids)

                        combined_results.append({
                            'Lipid': combined_lipid,
                            'Parent_Ion': parent_ions_str,  # Aggregated Parent_Ion values
                            'group_by_lipid': group_by_lipids_str,  # Aggregated group_by_lipid values
                            'Retention_Time': retention_times_str,
                            'Copy': np.nan,
                            'total_n_values': num_n_values  # Count of n_values in the cluster
                        })
                else:
                    # For total_positions_number equal to 1, do not generate combinations or add 'Copy' column
                    # Collect all n_values even if they exceed expected_positions
                    n_values_str = ' '.join([f"n-{n}" for n in n_values])
                    combined_lipid = f"{fa} {n_values_str}".strip()
                    retention_times = [str(n_values_times[n]) for n in n_values]
                    retention_times_str = ','.join(retention_times)
                    parent_ions = [n_values_parent_ions[n] for n in n_values]
                    parent_ions_str = separator.join(parent_ions)
                    group_by_lipids = [n_values_group_by_lipid[n] for n in n_values]
                    group_by_lipids_str = separator.join(group_by_lipids)

                    combined_results.append({
                        'Lipid': combined_lipid,
                        'Parent_Ion': parent_ions_str,  # Aggregated Parent_Ion values
                        'group_by_lipid': group_by_lipids_str,  # Aggregated group_by_lipid values
                        'Retention_Time': retention_times_str,
                        'Copy': np.nan,
                        'total_n_values': num_n_values  # Count of n_values in the cluster
                    })

    # Create a new DataFrame from the combined results
    df_combined_n_values = pd.DataFrame(combined_results)

    # Reorder columns to include 'Parent_Ion', 'group_by_lipid', and 'total_n_values'
    df_combined_n_values = df_combined_n_values[['Lipid', 'Parent_Ion', 'group_by_lipid', 'Retention_Time', 'Copy', 'total_n_values']]

    # Replace empty strings in 'Copy' column with NaN
    df_combined_n_values['Copy'] = df_combined_n_values['Copy'].replace('', np.nan)

    return df_combined_n_values

# Example usage:
# Assuming you have a DataFrame named 'master2' with appropriate columns including 'Parent_Ion' and 'group_by_lipid'
# df_combined_n_values = n_value_combine(master2)
# print(df_combined_n_values.head(60))

# Apply the function
# Replace 'master2' with your actual DataFrame variable
# df_combined_n_values = n_value_combine(master2)
# df_combined_n_values.to_excel('Projects/AMP/notpossible_9/hippo_WT_m1_FAD245_n_values_combined_nov17.xlsx', index=False)

# Display the first 60 rows
# df_combined_n_values.head(60)

# Apply the function
df_combined_n_values = n_value_combine(master2)
df_combined_n_values.to_excel('Projects/AMP/notpossible_9/hippo_WT_m1_FAD245_n_values_combined_nov18.xlsx', index=False)

# Display the first 60 rows
df_combined_n_values.head(60)


,Lipid,Parent_Ion,group_by_lipid,Retention_Time,Copy,total_n_values
0,FA(12:2) n-4 <x>,325.2,22,3.5,NaN,1
1,FA(15:3) n-8 <xx>,309.1,69,3.4,NaN,1
2,FA(16:2) n-10 <x>,297.2,101,6.2,NaN,1
3,FA(18:2) n-6 n-10,379.3| 323.2,149| 143,"7.5,7.5",NaN,2
4,FA(18:2) n-6 n-7,379.3| 365.2,149| 150,"7.7,7.7",Copy1,3
5,FA(18:2) n-6 n-10,379.3| 323.2,149| 143,"7.7,7.8",Copy2,3
6,FA(18:2) n-7 n-10,365.2| 323.2,150| 143,"7.7,7.8",Copy3,3
7,FA(18:2) n-9 n-10,339.3| 325.2,166| 152,"7.5,7.4",NaN,2
8,FA(18:2) n-4 n-6,409.3| 381.3,161| 163,"7.7,7.8",Copy1,7
9,FA(18:2) n-4 n-7,409.3| 367.3,161| 164,"7.7,7.8",Copy2,7
